In [19]:
import pandas as pd
import boto3
from io import StringIO,BytesIO
from datetime import datetime,timedelta  

Reading the object from s3 bucker

In [20]:
# Adapter Layer
def read_to_csv_df(bucket, key, columns=None, sep=',', decoding='utf-8'):
    csv_obj = bucket.Object(key=key).get().get('Body').read().decode(decoding)
    data = StringIO(csv_obj)
    df = pd.read_csv(data, delimiter=sep)
    if columns:
        df = df[columns]
    return df  
def write_df_to_s3(df, key, bucket):
    out_buffer = BytesIO()
    df.to_parquet(out_buffer, index=False)
    bucket.put_object(Body=out_buffer.getvalue(), Key=key)
    return True
    
def write_df_to_s3_csv(bucket, df, key):
    out_buffer = StringIO()
    df.to_csv(out_buffer, index=False)
    bucket.put_object(Body=out_buffer.getvalue(), Key=key)
    return True

def list_filter_in_prefix(bucket,prefix):
    files = [obj.key for obj in bucket.objects.filter(Prefix=prefix)]
    return files   


In [21]:
#Application Layer

def extrack(date_list,bucket):
    files=[key for date in date_list for key in list_filter_in_prefix(bucket,date)]
    df=pd.concat([read_to_csv_df(bucket,obj) for obj in files],ignore_index=True)
    return df


def Transfor(df,arg_date,columns):
    df['opening_price']=df.sort_values('Date').groupby(['ISIN','Date'])['StartPrice'].transform('first')
    df['closing_price']=df.sort_values('Date').groupby(['ISIN','Date'])['EndPrice'].transform('first')
    df=df.groupby(['ISIN','Date'],as_index=False).agg(opening_price_eur=('opening_price','min'),closing_price_eur=('closing_price','min'),MaxPrice_eur=('MaxPrice','max'),MinPrice_eur=('MinPrice','min'),TradedVolume=('TradedVolume','sum'))
    df['prev_closing_price']=df.sort_values(by=['Date']).groupby(['ISIN'])['closing_price_eur'].shift(1)
    df['change_prev_closing_%']=(df['closing_price_eur'] - df['prev_closing_price'])/df['prev_closing_price']*100
    df.drop(columns='prev_closing_price',inplace=True)
    df=df.round(decimals=2)
    df = df[df.Date >= arg_date]
    return df

    
def load(trg_key,bucket,trg_format,df,meta_key,extract_date_list,scr_format):
    key=trg_key+ datetime.today().strftime('%Y%m%d_%H%M%S')+trg_format
    write_df_to_s3(df,key,bucket)
    meta_file_update(bucket,meta_key,extract_date_list,scr_format)
    return True

def etl_report1(src_bucket, trg_bucket, arg_date, date_list, columns, trg_key, trg_format,meta_key):
    df =extrack(date_list,src_bucket)
    df = Transfor(df, arg_date, columns)
    extract_date_list = [date for date in date_list if date >= arg_date]
    load(trg_key, trg_bucket, trg_format, df,meta_key,extract_date_list,scr_format='%Y-%m-%d')
    return True

In [25]:
# Application Layer - not core
def retrun_date_list(bucket, arg_date, scr_format,meta_key):
    min_date = datetime.strptime(arg_date, scr_format).date() - timedelta(days=1)
    today=datetime.today().date()
    try:
        df_meta=read_to_csv_df(bucket, meta_key)
        dates = [(min_date + timedelta(days=x)) for x in range (0,(today-min_date).days+1 )]
        scr_date=set(pd.to_datetime(df_meta['source_date']).dt.date)
        date_missing=set(dates[1:])-scr_date
        if date_missing:
            min_date=min(set(dates[1:])-scr_date) - timedelta(days=1)
            return_dates=[date.strftime(scr_format) for date in dates if date>=min_date]
            return_min_dates=(min_date+timedelta(days=1)).strftime(scr_format)
        else:
            return_dates=[]
            return_min_dates=datetime(2200,1,1).date()
    except bucket.session.client('s3').execptions.NoSuchKey:
        return_dates = [(min_date + timedelta(days=x)).strftime(scr_format) for x in range(0, (today-min_date).days + 1)]
        return_min_date = arg_date
    return return_min_dates,return_dates


def meta_file_update(bucket,meta_key,extract_date_list,scr_format):
    df_new=pd.DataFrame(columns=['source_date','datetime_of_processing'])
    df_new['source_date']=extract_date_list
    df_new['datetime_of_processing']=datetime.today().strftime(scr_format)
    df_old= read_to_csv_df(bucket,meta_key)
    df_all=pd.concat([df_old,df_new])
    write_df_to_s3_csv(bucket,df_all,meta_key)
   



In [26]:
# main function entrypoint
def main():
    # Parameters/Configurations
    # Later read config
    arg_date = '2022-01-27'
    scr_format='%Y-%m-%d'
    meta_key='Meta_File.csv'
    scr_bucket='fezi-xetra-123'
    trg_bucket='fezi-xetra'
    columns=['ISIN', 'Date', 'Time', 'StartPrice', 'MaxPrice', 'MinPrice','EndPrice','TradedVolume']
    key='xetra_daily_report'+ datetime.today().strftime('%Y%m%d_%H%M%S')+'.parquet'
    trg_format='.parquet'
    trg_key='xetra_daily_report'        
    # Init
    s3=boto3.resource('s3')
    bucket_src=s3.Bucket(scr_bucket)
    bucket_trg=s3.Bucket(trg_bucket)
    # run application
    extract_date,date_list =retrun_date_list(bucket_trg, arg_date, scr_format,meta_key)
    etl_report1(bucket_src, bucket_trg, extract_date, date_list, columns, trg_key, trg_format, meta_key)

In [27]:
main()

C:\Users\Sam\AppData\Local\Temp\ipykernel_5604\1881802185.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df=pd.concat([read_to_csv_df(bucket,obj) for obj in files],ignore_index=True)


check upload

In [28]:
trg_bucket='fezi-xetra'
s3=boto3.resource('s3')
bucket_trg=s3.Bucket(trg_bucket)
for obj in bucket_trg.objects.all():
    print(obj.key)


Meta_File.csv
xetra_daily_report20250527_194655.parquet
xetra_daily_report20250528_011725.parquet
xetra_daily_report20250528_200611.parquet
xetra_daily_report20250528_202502.parquet
xetra_daily_report20250529_000653.parquet
xetra_daily_report20250530_035155.parquet
xetra_daily_report20250603_021301.parquet
xetra_daily_report20250603_025145.parquet
xetra_daily_report20250603_031817.parquet
xetra_daily_report20250603_035543.parquet
xetra_daily_report20250603_042128.parquet
xetra_daily_report20250603_044803.parquet
xetra_daily_report20250603_052414.parquet
xetra_daily_report20250603_155935.parquet
xetra_daily_report20250603_173456.parquet
xetra_daily_report20250604_005016.parquet
xetra_daily_report20250604_005840.parquet
xetra_daily_report20250604_122310.parquet
xetra_daily_report20250702_045901.parquet
xetra_daily_report20250702_050621.parquet
xetra_daily_report20250702_051416.parquet


In [29]:
prq_obj = bucket_trg.Object(key='xetra_daily_report20250702_051416.parquet').get()['Body'].read()
data = BytesIO(prq_obj)
df_prq = pd.read_parquet(data)


In [31]:
df_prq

,ISIN,Date,opening_price_eur,closing_price_eur,MaxPrice_eur,MinPrice_eur,TradedVolume,change_prev_closing_%
0,AT000000STR1,2022-01-27,37.90,37.90,37.90,37.00,485,1.34
1,AT000000STR1,2022-01-28,37.00,37.00,38.05,37.00,456,-2.37
2,AT000000STR1,2022-01-31,37.70,37.70,37.80,37.65,1492,1.89
3,AT00000FACC2,2022-01-27,7.64,7.64,7.64,7.62,60,-1.67
4,AT00000FACC2,2022-01-28,7.66,7.66,7.66,7.52,610,0.26
...,...,...,...,...,...,...,...,...
9538,XS2314660700,2022-01-28,20.05,20.05,20.33,20.05,58,-2.00
9539,XS2314660700,2022-01-31,20.61,20.61,20.61,20.17,846,2.77
9540,XS2376095068,2022-01-27,33.13,33.13,33.33,32.22,11504,-1.83
9541,XS2376095068,2022-01-28,33.23,33.23,33.23,32.77,0,0.31
